In [1]:
import pandas as pd

In [2]:
def float_round(x):
    if type(x) == float:
        return round(x, 3)
    else:
        return x

In [3]:
df = pd.read_csv('results/results_final.csv')

df.head()

,Model,Predictor,Alpha,Perturbation,Top1,Top5,Coverage,Size
0,Inception,Fixed,0.05,clean,0.755,0.984,0.962,2.821
1,Inception,Fixed,0.05,gaussian_noise,0.105,0.517,0.539,5.312
2,Inception,Fixed,0.05,blur,0.644,0.960,0.946,3.630
3,Inception,Fixed,0.05,brightness,0.659,0.965,0.938,3.266
4,Inception,Fixed,0.05,erasure,0.228,0.780,0.598,3.287


In [4]:
perturb_agg = df.groupby(['Perturbation'], as_index = False)[['Top1', 'Top5', 'Coverage', 'Size']].mean().sort_values('Top1', ascending = False)

perturb_agg = perturb_agg.map(float_round)

perturb_agg.to_csv('results/perturb_agg.csv', index = False)

perturb_agg

,Perturbation,Top1,Top5,Coverage,Size
2,clean,0.501,0.909,0.928,5.178
4,gaussian_noise,0.415,0.851,0.872,5.486
1,brightness,0.407,0.857,0.910,5.802
3,erasure,0.332,0.824,0.852,5.661
5,jpeg,0.320,0.802,0.870,6.057
0,blur,0.311,0.789,0.858,5.842


In [5]:
df_clean = df[df['Perturbation'] == 'clean']
df_perturb = df[~df.index.isin(df_clean.index)]

model_agg_clean = df_clean.groupby(['Model'], as_index = False)[['Top1', 'Top5', 'Coverage', 'Size']].mean().sort_values('Top1', ascending = False)

model_agg_clean = model_agg_clean.map(float_round)

model_agg_clean.to_csv('results/model_agg_clean.csv', index = False)

model_agg_clean

,Model,Top1,Top5,Coverage,Size
1,Inception,0.755,0.984,0.945,2.465
8,VGG16,0.579,0.954,0.927,4.021
0,DenseNet161,0.514,0.925,0.929,5.048
2,ResNeXt101,0.492,0.908,0.926,5.264
6,ResNet50,0.475,0.904,0.927,5.564
4,ResNet152,0.464,0.895,0.927,5.649
3,ResNet101,0.434,0.895,0.925,5.704
5,ResNet18,0.422,0.880,0.927,6.016
7,ShuffleNet,0.376,0.833,0.922,6.872


In [6]:
model_agg = df_perturb.groupby(['Model'], as_index = False)[['Top1', 'Top5', 'Coverage', 'Size']].mean()

# Reuse the clean-model ranking order for perturbed-model aggregation output.
model_order = model_agg_clean['Model'].tolist()
model_agg['Model'] = pd.Categorical(model_agg['Model'], categories = model_order, ordered = True)
model_agg = model_agg.sort_values('Model').reset_index(drop = True)

model_agg = model_agg.map(float_round)

model_agg.to_csv('results/model_agg.csv', index = False)

model_agg

,Model,Top1,Top5,Coverage,Size
0,Inception,0.365,0.774,0.711,3.852
1,VGG16,0.431,0.883,0.885,4.878
2,DenseNet161,0.385,0.857,0.884,5.548
3,ResNeXt101,0.376,0.845,0.895,5.871
4,ResNet50,0.374,0.850,0.902,6.033
5,ResNet152,0.363,0.840,0.894,5.949
6,ResNet101,0.339,0.829,0.881,5.920
7,ResNet18,0.315,0.813,0.894,6.381
8,ShuffleNet,0.267,0.729,0.903,7.491


In [7]:
conformal_agg = df.groupby(['Predictor'], as_index = False)[['Coverage', 'Size']].mean().sort_values('Coverage', ascending = False)

conformal_agg = conformal_agg.map(float_round)

conformal_agg

conformal_agg.to_csv('results/conformal_agg.csv', index = False)

In [8]:
accs = df.groupby(['Model', 'Perturbation'], as_index = False)[['Top1', 'Top5']].mean().sort_values('Model')

accs = accs.map(float_round)
accs.columns = pd.MultiIndex.from_tuples([
    ('Model', ''),
    ('Perturbation', ''),
    ('Accuracy', 'Top1'),
    ('Accuracy', 'Top5'),
])

accs.head()

Model    Perturbation Accuracy       
                                   Top1   Top5
0  DenseNet161            blur    0.286  0.790
1  DenseNet161      brightness    0.417  0.877
2  DenseNet161           clean    0.514  0.925
3  DenseNet161         erasure    0.377  0.856
4  DenseNet161  gaussian_noise    0.482  0.913

In [9]:
pivot = df.pivot_table(index = ['Model', 'Perturbation'], values = ['Coverage', 'Size'], columns = 'Predictor').reset_index()

pivot = pivot.map(float_round)

pivot.head(10)

Model    Perturbation Coverage                        Size  \
Predictor                                   APS  Fixed  Naive   RAPS    APS   
0          DenseNet161            blur    0.812  0.812  0.813  0.796  5.500   
1          DenseNet161      brightness    0.909  0.909  0.909  0.893  5.687   
2          DenseNet161           clean    0.929  0.929  0.931  0.928  5.055   
3          DenseNet161         erasure    0.900  0.903  0.903  0.885  5.815   
4          DenseNet161  gaussian_noise    0.919  0.921  0.920  0.916  5.147   
5          DenseNet161            jpeg    0.893  0.893  0.893  0.875  5.874   
6            Inception            blur    0.922  0.922  0.925  0.903  3.178   
7            Inception      brightness    0.913  0.912  0.914  0.903  2.866   
8            Inception           clean    0.946  0.946  0.948  0.941  2.479   
9            Inception         erasure    0.540  0.537  0.543  0.543  2.853   

                                
Predictor  Fixed  Naive   RAPS  
0          5.510  5.513  5.248  
1          5.694  5.699  5.396  
2          5.068  5.067  5.001  
3          5.824  5.830  5.533  
4          5.152  5.153  5.083  
5          5.885  5.889  5.529  
6          3.180  3.216  2.850  
7          2.869  2.899  2.692  
8          2.482  2.506  2.394  
9          2.858  2.896  2.778

In [10]:
merged = pivot.merge(
    accs,
    on = [('Model', ''), ('Perturbation', '')],
    how = 'left',
)

merged.sort_values(['Perturbation', 'Model']).head(10)

merged.to_csv('results/table.csv', index = False)